# Notebook 09 — Informed Compression (Synthesis)

Use mechanistic insights from notebooks 04-08 to design an informed compression
strategy that outperforms blind uniform SVD.

**Pipeline:**
1. Load all previous analysis results (probing, patching, head ablation, neuron analysis)
2. Design per-layer, per-component SVD rank allocations
3. Grand comparison: Uniform vs Adaptive (energy) vs Informed vs Greedy
4. Pareto frontier across all strategies
5. Fine-tuning recovery analysis
6. Inference benchmarks (CPU + GPU)
7. Final summary

**Structure:** Setup → A (load analyses) → B (design strategies) →
C (grand comparison) → D (per-emotion impact) → E (fine-tuning recovery) →
F (inference benchmarks) → G (export).

## Setup

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'
    os.chdir(PROJECT_ROOT)
    sys.path.insert(0, PROJECT_ROOT)
    !pip install -q transformers datasets scikit-learn
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    sys.path.insert(0, PROJECT_ROOT)

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import pickle
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from transformers import Trainer, TrainingArguments

from src.data import load_goemotions
from src.models import load_bert_classifier
from src.compression import (
    apply_svd_compression, get_target_layer_names, filter_layer_names,
    compute_singular_value_energy, compute_adaptive_ranks,
    count_parameters, get_compression_ratio,
)
from src.utils import compute_metrics

EXCLUDED_EMOTIONS = ['neutral', 'grief', 'nervousness', 'pride', 'relief']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

results_dir = os.path.join(PROJECT_ROOT, 'results')
out_dir = os.path.join(results_dir, 'notebook9')
os.makedirs(out_dir, exist_ok=True)
print(f'Output directory: {out_dir}')

In [ ]:
# Load data + baseline model
train_ds, val_ds, test_ds, emotion_names, data_collator = load_goemotions(exclude_emotions=EXCLUDED_EMOTIONS)

MODEL_PATH = os.path.join(PROJECT_ROOT, 'results', 'bert-goemotions-23emo-final')
baseline_model = load_bert_classifier(model_name=MODEL_PATH, num_labels=len(emotion_names))
baseline_model.to(device)
baseline_model.eval()

num_labels = len(emotion_names)
print(f'Baseline model loaded')
print(f'num_labels: {num_labels}')
print(f'Test set: {len(test_ds)} samples')

## Part A — Load Mechanistic Analysis Results

Pulls CSVs/pickles produced by notebooks 04 (probing), 05 (patching),
06 (head ablation) and 07 (neuron analysis). Searches the new
`results/notebook{N}/` subdirectories first and falls back to the flat
`results/` directory for backward compatibility.

In [ ]:
def _first_existing(*paths):
    """Return the first existing path from the list, or None."""
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None


# Probing (NB04)
probe_path = _first_existing(
    os.path.join(results_dir, 'notebook4', 'crystallization_layers.csv'),
    os.path.join(results_dir, 'crystallization_layers.csv'),
)
probe_results_path = _first_existing(
    os.path.join(results_dir, 'notebook4', 'probe_results.csv'),
    os.path.join(results_dir, 'probe_results.csv'),
)
try:
    crystallization_df = pd.read_csv(probe_path) if probe_path else None
    probe_results = pd.read_csv(probe_results_path) if probe_results_path else None
    if crystallization_df is not None:
        print(f'Probing results: {len(crystallization_df)} emotions  '
              f'(mean crystallization layer = {crystallization_df["crystallization_layer"].mean():.1f})')
    else:
        print('WARNING: Probing results not found. Run notebook 04 first.')
except Exception as e:
    print(f'Probing load failed: {e}')
    crystallization_df = None
    probe_results = None

# Head ablation (NB06)
head_imp_path = _first_existing(
    os.path.join(results_dir, 'notebook6', 'head_importance_matrix.npy'),
    os.path.join(results_dir, 'head_importance_matrix.npy'),
)
head_cat_path = _first_existing(
    os.path.join(results_dir, 'notebook6', 'head_categories.csv'),
    os.path.join(results_dir, 'head_categories.csv'),
)
layer_attn_path = _first_existing(
    os.path.join(results_dir, 'notebook6', 'layer_attention_importance.csv'),
    os.path.join(results_dir, 'layer_attention_importance.csv'),
)
try:
    head_importance = np.load(head_imp_path) if head_imp_path else None
    head_categories = pd.read_csv(head_cat_path) if head_cat_path else None
    layer_attn_importance = pd.read_csv(layer_attn_path) if layer_attn_path else None
    if head_importance is not None:
        print(f'Head ablation results: matrix shape {head_importance.shape}')
    else:
        print('WARNING: Head ablation results not found. Run notebook 06 first.')
except Exception as e:
    print(f'Head ablation load failed: {e}')
    head_importance = head_categories = layer_attn_importance = None

# Activation patching (NB05)
patching_path = _first_existing(
    os.path.join(results_dir, 'notebook5', 'activation_patching_per_component.csv'),
    os.path.join(results_dir, 'activation_patching_per_component.csv'),
)
try:
    patching_results = pd.read_csv(patching_path) if patching_path else None
    if patching_results is not None:
        print(f'Patching results: {len(patching_results)} rows  '
              f'(components: {patching_results["component"].unique().tolist()})')
    else:
        print('WARNING: Patching results not found. Run notebook 05 first.')
except Exception as e:
    print(f'Patching load failed: {e}')
    patching_results = None

# Neuron selectivity (NB07)
neuron_path = _first_existing(
    os.path.join(results_dir, 'notebook7', 'neuron_selectivity.pkl'),
    os.path.join(results_dir, 'neuron_selectivity.pkl'),
)
try:
    if neuron_path:
        with open(neuron_path, 'rb') as f:
            neuron_selectivity = pickle.load(f)
        print(f'Neuron selectivity: {len(neuron_selectivity)} layers, '
              f'each shape {neuron_selectivity[0].shape}')
    else:
        neuron_selectivity = None
        print('WARNING: Neuron results not found. Run notebook 07 first.')
except Exception as e:
    print(f'Neuron load failed: {e}')
    neuron_selectivity = None

## Part B — Design Compression Strategies

Strategy families compared:
- **Uniform** — same rank for every layer (baseline)
- **Adaptive** — per-layer ranks from singular-value energy thresholds
- **Informed V1** — linear importance-to-rank mapping using signals from A
- **Greedy (data-driven)** — uses NB03 empirical sensitivity data to build a
  priority queue of compression moves sorted by efficiency (params saved / F1 cost)

### B.1 — Layer importance (for Informed V1)

In [ ]:
def compute_layer_importance_scores(crystallization_df=None, layer_attn_importance=None,
                                     patching_results=None, neuron_selectivity=None):
    """Compute composite attention/FFN importance per layer from all analyses.

    Returns (attn_importance, ffn_importance) each shape (12,), normalized to [0, 1].
    """
    n_layers = 12
    attn_scores = np.ones(n_layers) * 0.5
    ffn_scores = np.ones(n_layers) * 0.5

    n_signals = 0

    # Signal 1: Probing — layers near crystallization are critical
    if crystallization_df is not None:
        crystal_layers = crystallization_df['crystallization_layer'].values
        crystal_counts = np.zeros(13)  # 0-12 including embedding
        for cl in crystal_layers:
            if not np.isnan(cl):
                crystal_counts[int(cl)] += 1
        encoder_crystal = crystal_counts[1:]  # (12,)
        before_crystal = np.zeros(n_layers)
        for cl in crystal_layers:
            if not np.isnan(cl) and int(cl) >= 2:
                before_crystal[int(cl) - 2] += 0.5
                before_crystal[int(cl) - 1] += 1.0
        combined = encoder_crystal + before_crystal
        if combined.max() > 0:
            probing_signal = combined / combined.max()
            attn_scores += probing_signal
            ffn_scores += probing_signal
            n_signals += 1
            print(f'  Probing signal: peak at layers {np.argsort(probing_signal)[::-1][:3]}')

    # Signal 2: Head ablation — layers with important heads
    if layer_attn_importance is not None:
        attn_signal = layer_attn_importance['mean_head_importance'].values
        if attn_signal.max() > 0:
            attn_signal = attn_signal / attn_signal.max()
            attn_scores += attn_signal * 1.5
            n_signals += 1
            print(f'  Head ablation signal: peak at layers {np.argsort(attn_signal)[::-1][:3]}')

    # Signal 3: Activation patching — layers with highest restoration
    if patching_results is not None:
        try:
            for component, target_scores in [('attention', attn_scores), ('ffn', ffn_scores)]:
                subset = patching_results[patching_results['component'] == component]
                if len(subset) > 0:
                    layer_restoration = subset.groupby('layer')['restoration_score'].mean()
                    layer_restoration = layer_restoration.reindex(range(n_layers), fill_value=0).values
                    if layer_restoration.max() > 0:
                        patch_signal = layer_restoration / layer_restoration.max()
                        target_scores += patch_signal
                        print(f'  Patching ({component}): peak at layers '
                              f'{np.argsort(patch_signal)[::-1][:3]}')
            n_signals += 1
        except Exception as e:
            print(f'  Patching signal skipped: {e}')

    # Signal 4: Neuron selectivity (FFN-only)
    if neuron_selectivity is not None:
        try:
            layer_selectivity = np.array([np.abs(sel).mean() for sel in neuron_selectivity])
            if layer_selectivity.max() > 0:
                neuron_signal = layer_selectivity / layer_selectivity.max()
                ffn_scores += neuron_signal * 1.5
                n_signals += 1
                print(f'  Neuron selectivity: peak at layers {np.argsort(neuron_signal)[::-1][:3]}')
        except Exception as e:
            print(f'  Neuron signal skipped: {e}')

    if attn_scores.max() > 0:
        attn_scores = attn_scores / attn_scores.max()
    if ffn_scores.max() > 0:
        ffn_scores = ffn_scores / ffn_scores.max()

    print(f'\n  Combined from {n_signals} signal sources')
    return attn_scores, ffn_scores


print('Computing layer importance from mechanistic analyses...')
attn_importance, ffn_importance = compute_layer_importance_scores(
    crystallization_df, layer_attn_importance, patching_results, neuron_selectivity
)

In [ ]:
# Visualize layer importance
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(12)
width = 0.35
ax.bar(x - width/2, attn_importance, width, label='Attention', color='steelblue', alpha=0.8)
ax.bar(x + width/2, ffn_importance, width, label='FFN', color='coral', alpha=0.8)

ax.set_xlabel('Layer')
ax.set_ylabel('Importance Score (normalized)')
ax.set_title('Layer Importance from Mechanistic Analyses')
ax.set_xticks(x)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'informed_layer_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

### B.2 — Informed V1 (linear importance → rank)

In [ ]:
def design_informed_ranks(attn_importance, ffn_importance,
                          target_compression_ratio=0.5,
                          min_rank=32, max_rank=700):
    """Linear importance-to-rank mapping. Higher importance -> higher rank."""
    temp_model = load_bert_classifier(model_name=MODEL_PATH, num_labels=num_labels)
    all_layers = get_target_layer_names(temp_model)
    del temp_model

    rank_dict = {}
    for layer_name in all_layers:
        layer_idx = int(layer_name.split('.')[3])
        is_attention = 'attention' in layer_name
        is_ffn = ('intermediate' in layer_name or
                  ('output.dense' in layer_name and 'attention' not in layer_name))

        if is_attention:
            importance = attn_importance[layer_idx]
        elif is_ffn:
            importance = ffn_importance[layer_idx]
        else:
            importance = 0.5

        rank = int(min_rank + importance * (max_rank - min_rank))
        rank = max(min_rank, min(max_rank, rank))
        rank_dict[layer_name] = rank
    return rank_dict


compression_configs = {
    'informed_light':      {'min_rank': 200, 'max_rank': 700},
    'informed_moderate':   {'min_rank': 64,  'max_rank': 512},
    'informed_aggressive': {'min_rank': 32,  'max_rank': 256},
}

informed_ranks = {}
for config_name, params in compression_configs.items():
    ranks = design_informed_ranks(
        attn_importance, ffn_importance,
        min_rank=params['min_rank'], max_rank=params['max_rank'],
    )
    informed_ranks[config_name] = ranks
    rank_values = list(ranks.values())
    print(f'\n{config_name}: min={min(rank_values)}, max={max(rank_values)}, '
          f'mean={np.mean(rank_values):.0f}, std={np.std(rank_values):.0f}')

### B.3 — Greedy (data-driven from NB03 sensitivity)

The linear V1 mapping ignores the empirical asymmetry between components: Q/K
can be compressed to rank 64 almost for free, while FFN Intermediate breaks
down at rank 256. The greedy algorithm enumerates every possible rank move
across components and depth bands, sorts them by **params-saved / F1-cost**,
and takes moves from most to least efficient until the target compression
ratio is reached.

| Component | Rank | F1 Retention | Verdict |
|-----------|------|-------------|---------|
| Q | 64 | 98.6% | Nearly free |
| K | 64 | 97.6% | Nearly free |
| V | 256 | 93.5% | Moderate cost |
| Attn Out | 256 | 92.2% | Moderate cost |
| FFN Out | 256 | 75.0% | Expensive |
| FFN Inter | 256 | 51.8% | Very expensive |

Late layers (8-11) bear 67.5% of total F1 cost; early layers (0-3) only 14.8%.

In [ ]:
def design_greedy_ranks(target_ratio, baseline_model, baseline_params,
                        max_cost_per_move=15.0):
    """Data-driven greedy allocation using NB03 component+depth sensitivity.

    Returns:
        rank_dict: {layer_name: rank} — only compressed layers included
        achieved_ratio: actual params ratio achieved
        total_cost: estimated total F1 cost in pp
        taken: list of applied moves (for analysis)
    """
    # Empirical F1 retention (%) when compressing ALL 12 layers of each component
    COMPONENT_RETENTION = {
        'query':       {256: 100.4, 128: 99.4, 64: 98.6},
        'key':         {256: 99.9,  128: 98.4, 64: 97.6},
        'value':       {256: 93.5,  128: 68.5},
        'attn_output': {256: 92.2,  128: 56.3},
        'ffn_inter':   {256: 51.8},
        'ffn_output':  {256: 75.0,  128: 22.2},
    }

    DEPTH_FRACTIONS = {'early': 0.148, 'middle': 0.177, 'late': 0.675}
    BAND_LAYERS = {'early': [0,1,2,3], 'middle': [4,5,6,7], 'late': [8,9,10,11]}

    COMPONENT_DIMS = {
        'query': (768,768), 'key': (768,768),
        'value': (768,768), 'attn_output': (768,768),
        'ffn_inter': (3072,768), 'ffn_output': (768,3072),
    }

    moves = []
    for comp, ret_data in COMPONENT_RETENTION.items():
        out_d, in_d = COMPONENT_DIMS[comp]
        available_ranks = sorted(ret_data.keys(), reverse=True)

        prev_rank = None
        prev_retention = 100.0

        for rank in available_ranks:
            retention = ret_data[rank]
            inc_cost_total = prev_retention - retention

            svd_params = rank * (out_d + in_d)
            if prev_rank is None:
                saved_per_layer = out_d * in_d - svd_params
            else:
                saved_per_layer = prev_rank * (out_d + in_d) - svd_params

            for band, layers in BAND_LAYERS.items():
                n = len(layers)
                frac = DEPTH_FRACTIONS[band]
                band_saved = n * saved_per_layer
                band_cost = inc_cost_total * frac

                eff = float('inf') if band_cost <= 0 else band_saved / band_cost

                moves.append({
                    'comp': comp, 'band': band,
                    'from_rank': prev_rank, 'to_rank': rank,
                    'saved': band_saved, 'cost': band_cost, 'eff': eff,
                })

            prev_rank = rank
            prev_retention = retention

    moves.sort(key=lambda m: (-1 if m['eff'] == float('inf') else 0, -m['eff']))

    state = {(c, b): None for c in COMPONENT_RETENTION for b in BAND_LAYERS}
    total_saved = 0
    total_cost = 0.0
    taken = []

    for m in moves:
        key = (m['comp'], m['band'])
        if state[key] != m['from_rank']:
            continue
        if m['cost'] > max_cost_per_move:
            continue

        state[key] = m['to_rank']
        total_saved += m['saved']
        total_cost += m['cost']
        taken.append(m)

        if (baseline_params - total_saved) / baseline_params <= target_ratio:
            break

    all_layers = get_target_layer_names(baseline_model)
    rank_dict = {}

    for name in all_layers:
        layer_idx = int(name.split('.')[3])
        band = 'early' if layer_idx <= 3 else ('middle' if layer_idx <= 7 else 'late')

        if   'attention.self.query' in name:    comp = 'query'
        elif 'attention.self.key' in name:      comp = 'key'
        elif 'attention.self.value' in name:    comp = 'value'
        elif 'attention.output.dense' in name:  comp = 'attn_output'
        elif 'intermediate.dense' in name:      comp = 'ffn_inter'
        else:                                    comp = 'ffn_output'

        rank = state.get((comp, band))
        if rank is not None:
            rank_dict[name] = rank

    achieved_ratio = (baseline_params - total_saved) / baseline_params
    return rank_dict, achieved_ratio, total_cost, taken


_greedy_params = count_parameters(baseline_model)['total']

greedy_ranks = {}
greedy_moves_log = {}
print('Greedy compression strategies (data-driven from NB03):\n')

for target in [0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50]:
    name = f'greedy_{int(target*100)}'
    rank_dict, achieved, est_cost, taken = design_greedy_ranks(
        target, baseline_model, _greedy_params
    )
    greedy_ranks[name] = rank_dict
    greedy_moves_log[name] = {
        'target_ratio': target,
        'achieved_ratio': achieved,
        'estimated_cost_pp': est_cost,
        'moves': taken,
    }

    n_compressed = len(rank_dict)
    print(f'{name} (target={target:.0%}, achieved~{achieved:.3f}):')
    print(f'  Est. F1 cost: {est_cost:.1f}pp | Compressed: {n_compressed}/72 layers')
    for m in taken:
        fr = str(m['from_rank']) if m['from_rank'] else 'full'
        print(f"    {m['comp']:12s} {m['band']:6s}: {fr:>4s}->{m['to_rank']:<4d}  "
              f"saved={m['saved']:>10,}  cost={m['cost']:+6.2f}pp")
    print()

## Part C — Grand Comparison (Uniform vs Adaptive vs Informed vs Greedy)

In [ ]:
@torch.no_grad()
def evaluate_model(model, dataset, data_collator, device, batch_size=64):
    """Return macro F1, micro F1, per-emotion F1 for a model."""
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=data_collator, shuffle=False)
    all_preds, all_labels = [], []

    model.eval()
    for batch in tqdm(loader, desc='Evaluating', leave=False):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.cpu()
        preds = (torch.sigmoid(logits) > 0.5).float()

        all_preds.append(preds)
        all_labels.append(labels)

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    micro_f1 = f1_score(all_labels, all_preds, average='micro', zero_division=0)
    per_emotion = [f1_score(all_labels[:, i], all_preds[:, i], zero_division=0)
                   for i in range(all_labels.shape[1])]

    return {
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
        'per_emotion_f1': np.array(per_emotion),
    }

In [ ]:
# Baseline evaluation
print('Evaluating baseline...')
baseline_results = evaluate_model(baseline_model, test_ds, data_collator, device)
baseline_params = count_parameters(baseline_model)['total']
print(f'Baseline: macro_f1={baseline_results["macro_f1"]:.4f}, '
      f'micro_f1={baseline_results["micro_f1"]:.4f}, params={baseline_params:,}')

In [ ]:
# Assemble full strategy list
strategies = []

for rank in [512, 384, 256, 128, 64, 32]:
    strategies.append({'name': f'uniform_r{rank}', 'type': 'uniform', 'rank': rank})

for threshold in [0.99, 0.95, 0.90, 0.80]:
    strategies.append({'name': f'adaptive_e{int(threshold*100)}', 'type': 'adaptive',
                       'threshold': threshold})

for config_name, ranks in informed_ranks.items():
    strategies.append({'name': config_name, 'type': 'informed', 'ranks': ranks})

for config_name, ranks in greedy_ranks.items():
    strategies.append({'name': config_name, 'type': 'greedy', 'ranks': ranks})

print(f'Total strategies to evaluate: {len(strategies)}')
for s in strategies:
    print(f'  - {s["name"]} ({s["type"]})')

In [ ]:
# Evaluate all strategies
all_results = []
all_layers = get_target_layer_names(baseline_model)

for strategy in tqdm(strategies, desc='Evaluating strategies'):
    print(f'\n--- {strategy["name"]} ---')

    if strategy['type'] == 'uniform':
        compressed = apply_svd_compression(
            baseline_model, rank=strategy['rank'], layer_names=all_layers,
        )
    elif strategy['type'] == 'adaptive':
        energy_info = compute_singular_value_energy(baseline_model)
        adaptive_ranks = compute_adaptive_ranks(energy_info, energy_threshold=strategy['threshold'])
        compressed = apply_svd_compression(
            baseline_model, rank=adaptive_ranks, layer_names=all_layers,
        )
    elif strategy['type'] in ('informed', 'greedy'):
        compressed = apply_svd_compression(
            baseline_model, rank=strategy['ranks'], layer_names=all_layers,
        )

    compressed.to(device)
    comp_params = count_parameters(compressed)['total']
    comp_ratio = comp_params / baseline_params

    results = evaluate_model(compressed, test_ds, data_collator, device)

    all_results.append({
        'strategy': strategy['name'],
        'type': strategy['type'],
        'params': comp_params,
        'compression_ratio': comp_ratio,
        'macro_f1': results['macro_f1'],
        'micro_f1': results['micro_f1'],
        'f1_retention': results['macro_f1'] / baseline_results['macro_f1'],
        'per_emotion_f1': results['per_emotion_f1'],
    })

    print(f'  params={comp_params:,} ({comp_ratio:.2%}), '
          f'macro_f1={results["macro_f1"]:.4f} '
          f'({all_results[-1]["f1_retention"]:.2%} retained)')

    del compressed
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# Summary DataFrame (drop per-emotion arrays for tabular display)
results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'per_emotion_f1'}
                           for r in all_results])
results_df = results_df.sort_values('compression_ratio')

print('\nAll Results:')
print(results_df[['strategy', 'type', 'compression_ratio',
                  'macro_f1', 'f1_retention']].to_string(index=False))

### C.1 — Pareto frontier

In [ ]:
def compute_pareto_frontier(df, x_col='compression_ratio', y_col='macro_f1'):
    """Points where lower x AND higher y than all previously seen."""
    pareto = []
    sorted_df = df.sort_values(x_col)
    best_y = -np.inf
    for _, row in sorted_df.iterrows():
        if row[y_col] > best_y:
            pareto.append(row)
            best_y = row[y_col]
    return pd.DataFrame(pareto)


fig, ax = plt.subplots(figsize=(12, 7))

colors = {'uniform': 'steelblue', 'adaptive': 'orange',
          'informed': 'crimson', 'greedy': '#9b59b6'}
markers = {'uniform': 'o', 'adaptive': 's', 'informed': '*', 'greedy': 'P'}
sizes = {'uniform': 80, 'adaptive': 100, 'informed': 200, 'greedy': 160}

for stype in ['uniform', 'adaptive', 'informed', 'greedy']:
    subset = results_df[results_df['type'] == stype]
    if len(subset) == 0:
        continue
    ax.scatter(
        subset['compression_ratio'], subset['macro_f1'],
        c=colors[stype], marker=markers[stype], s=sizes[stype],
        label=stype.replace('_', ' ').title(), alpha=0.8,
        edgecolors='white', linewidth=0.5, zorder=3,
    )
    for _, row in subset.iterrows():
        label = row['strategy']
        for prefix in ['uniform_', 'adaptive_', 'greedy_', 'informed_']:
            label = label.replace(prefix, '')
        ax.annotate(label, (row['compression_ratio'], row['macro_f1']),
                    fontsize=7, ha='center', va='bottom',
                    textcoords='offset points', xytext=(0, 5))

pareto_df = compute_pareto_frontier(results_df)
pareto_sorted = pareto_df.sort_values('compression_ratio')
ax.plot(pareto_sorted['compression_ratio'], pareto_sorted['macro_f1'],
        'k--', alpha=0.4, label='Pareto frontier', zorder=1)

ax.axhline(y=baseline_results['macro_f1'], color='green', linestyle=':',
           alpha=0.5, label='Baseline')

ax.set_xlabel('Parameter Ratio (compressed / original)')
ax.set_ylabel('Macro F1')
ax.set_title('Compression Strategy Comparison — Pareto Frontier')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'pareto_frontier_all_strategies.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('\nPareto-optimal strategies:')
for _, row in pareto_sorted.iterrows():
    print(f'  {row["strategy"]:25s} ratio={row["compression_ratio"]:.3f}  '
          f'F1={row["macro_f1"]:.4f}  ({row["type"]})')

## Part D — Per-Emotion Impact Comparison

In [ ]:
def find_closest_strategy(results, strategy_type, target_ratio):
    subset = [r for r in results if r['type'] == strategy_type]
    if not subset:
        return None
    return min(subset, key=lambda r: abs(r['compression_ratio'] - target_ratio))


target_ratio = 0.80
comparable = {}
for stype in ['uniform', 'adaptive', 'greedy']:
    result = find_closest_strategy(all_results, stype, target_ratio)
    if result:
        comparable[stype] = result
        print(f'{stype}: {result["strategy"]} (ratio={result["compression_ratio"]:.3f}, '
              f'F1={result["macro_f1"]:.4f})')

per_emotion_comparison_rows = []

if len(comparable) >= 2:
    fig, ax = plt.subplots(figsize=(16, 6))

    x = np.arange(len(emotion_names))
    width = 0.25

    for i, (stype, result) in enumerate(comparable.items()):
        f1_drop = baseline_results['per_emotion_f1'] - result['per_emotion_f1']
        ax.bar(x + i * width - width, f1_drop, width,
               label=f'{stype} ({result["strategy"]})',
               color=colors.get(stype, 'gray'), alpha=0.8)
        for emo_idx, emo in enumerate(emotion_names):
            per_emotion_comparison_rows.append({
                'emotion': emo,
                'strategy_type': stype,
                'strategy_name': result['strategy'],
                'compression_ratio': result['compression_ratio'],
                'baseline_f1': baseline_results['per_emotion_f1'][emo_idx],
                'compressed_f1': result['per_emotion_f1'][emo_idx],
                'f1_drop': f1_drop[emo_idx],
            })

    ax.set_xlabel('Emotion')
    ax.set_ylabel('F1 Drop (positive = degradation)')
    ax.set_title('Per-Emotion F1 Degradation by Compression Strategy')
    ax.set_xticks(x)
    ax.set_xticklabels(emotion_names, rotation=45, ha='right', fontsize=8)
    ax.legend()
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'per_emotion_strategy_comparison.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

## Part E — Fine-Tuning Recovery Analysis

Can a few epochs of fine-tuning recover performance lost to compression? The
recovery rate itself is informative: **distributed** emotions recover, **concentrated**
emotions don't.

In [ ]:
def finetune_compressed(compressed_model, train_ds, val_ds, data_collator,
                        epochs=3, lr=2e-5, batch_size=32, output_dir=None):
    """Fine-tune a compressed model for a few epochs."""
    if output_dir is None:
        output_dir = os.path.join(PROJECT_ROOT, 'results', 'finetuned_compressed_tmp')

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=64,
        learning_rate=lr,
        warmup_ratio=0.1,
        eval_strategy='epoch',
        save_strategy='no',
        logging_steps=100,
        report_to='none',
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=0,
    )

    trainer = Trainer(
        model=compressed_model, args=training_args,
        train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=data_collator, compute_metrics=compute_metrics,
    )

    trainer.train()
    return compressed_model

In [ ]:
# Pick best greedy config (>=90% retention, most compressed)
greedy_results = [r for r in all_results if r['type'] == 'greedy']
greedy_above_90 = [r for r in greedy_results if r['f1_retention'] > 0.90]
if greedy_above_90:
    best_greedy = min(greedy_above_90, key=lambda r: r['compression_ratio'])
else:
    best_greedy = max(greedy_results, key=lambda r: r['macro_f1'])

ft_strategy_name = best_greedy['strategy']
ft_ranks = greedy_ranks[ft_strategy_name]

print(f'Fine-tuning {ft_strategy_name}...')
rank_values = list(ft_ranks.values())
print(f'Rank range: {min(rank_values)} - {max(rank_values)}, mean={np.mean(rank_values):.0f}')
print(f'Pre-FT: macro_f1={best_greedy["macro_f1"]:.4f}, ratio={best_greedy["compression_ratio"]:.3f}')

# Fresh compressed model for fine-tuning
ft_model = apply_svd_compression(
    baseline_model, rank=ft_ranks, layer_names=get_target_layer_names(baseline_model),
)

ft_model.to(device)
pre_ft_results = evaluate_model(ft_model, test_ds, data_collator, device)
print(f'\nBefore fine-tuning: macro_f1={pre_ft_results["macro_f1"]:.4f}')

In [ ]:
ft_model = finetune_compressed(
    ft_model, train_ds, val_ds, data_collator,
    epochs=3, lr=2e-5,
)

ft_model.to(device)
post_ft_results = evaluate_model(ft_model, test_ds, data_collator, device)
print(f'\nAfter fine-tuning: macro_f1={post_ft_results["macro_f1"]:.4f}')
print(f'Recovery: {pre_ft_results["macro_f1"]:.4f} -> {post_ft_results["macro_f1"]:.4f} '
      f'(baseline={baseline_results["macro_f1"]:.4f})')

In [ ]:
# Per-emotion recovery
recovery_df = pd.DataFrame({
    'emotion': emotion_names,
    'baseline_f1': baseline_results['per_emotion_f1'],
    'compressed_f1': pre_ft_results['per_emotion_f1'],
    'finetuned_f1': post_ft_results['per_emotion_f1'],
})

recovery_df['compression_drop'] = recovery_df['baseline_f1'] - recovery_df['compressed_f1']
recovery_df['ft_recovery']      = recovery_df['finetuned_f1'] - recovery_df['compressed_f1']
recovery_df['residual_gap']     = recovery_df['baseline_f1'] - recovery_df['finetuned_f1']
recovery_df['recovery_rate']    = np.where(
    recovery_df['compression_drop'] > 0.01,
    recovery_df['ft_recovery'] / recovery_df['compression_drop'],
    np.nan,
)

print('\nPer-Emotion Recovery Analysis:')
print(recovery_df[['emotion', 'baseline_f1', 'compressed_f1', 'finetuned_f1',
                   'recovery_rate']].to_string(index=False))

In [ ]:
# Recovery visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

x = np.arange(len(emotion_names))
width = 0.25

axes[0].bar(x - width, recovery_df['baseline_f1'],   width, label='Baseline',   color='green', alpha=0.7)
axes[0].bar(x,         recovery_df['compressed_f1'], width, label='Compressed', color='red',   alpha=0.7)
axes[0].bar(x + width, recovery_df['finetuned_f1'],  width, label='Fine-tuned', color='blue',  alpha=0.7)
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('F1 Score')
axes[0].set_title('Per-Emotion: Baseline vs Compressed vs Fine-tuned')
axes[0].set_xticks(x)
axes[0].set_xticklabels(emotion_names, rotation=45, ha='right', fontsize=7)
axes[0].legend(fontsize=8)

valid_recovery = recovery_df.dropna(subset=['recovery_rate']).sort_values('recovery_rate')
colors_recovery = ['red' if r < 0.5 else 'orange' if r < 0.8 else 'green'
                   for r in valid_recovery['recovery_rate']]
axes[1].barh(range(len(valid_recovery)), valid_recovery['recovery_rate'].clip(-0.5, 1.5),
             color=colors_recovery, alpha=0.8)
axes[1].set_yticks(range(len(valid_recovery)))
axes[1].set_yticklabels(valid_recovery['emotion'], fontsize=8)
axes[1].axvline(x=1.0, color='green', linestyle='--', alpha=0.5, label='Full recovery')
axes[1].axvline(x=0.0, color='red',   linestyle='--', alpha=0.5, label='No recovery')
axes[1].set_xlabel('Recovery Rate')
axes[1].set_title('Fine-Tuning Recovery Rate per Emotion')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'finetuning_recovery.png'), dpi=150, bbox_inches='tight')
plt.show()

low_recovery  = valid_recovery[valid_recovery['recovery_rate'] < 0.5]
high_recovery = valid_recovery[valid_recovery['recovery_rate'] > 0.8]
print(f'\nEmotions with LOW recovery (<50%): {list(low_recovery["emotion"])}')
print(f'  -> These have CONCENTRATED representations (compression is destructive)')
print(f'\nEmotions with HIGH recovery (>80%): {list(high_recovery["emotion"])}')
print(f'  -> These have DISTRIBUTED representations (compression is recoverable)')

## Part F — Inference Benchmarks

In [ ]:
def benchmark_inference(model, tokenizer_name='bert-base-uncased',
                        n_warmup=10, n_runs=50, batch_size=1, seq_len=128, device='cpu'):
    """Return latency percentiles and throughput (samples/sec)."""
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

    dummy_text = 'This is a test sentence for benchmarking inference speed.' * 5
    inputs = tokenizer(dummy_text, return_tensors='pt', padding='max_length',
                       truncation=True, max_length=seq_len)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    if batch_size > 1:
        inputs = {k: v.expand(batch_size, -1) for k, v in inputs.items()}

    model.to(device)
    model.eval()

    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(**inputs)

    if device != 'cpu' and torch.cuda.is_available():
        torch.cuda.synchronize()

    latencies = []
    with torch.no_grad():
        for _ in range(n_runs):
            start = time.perf_counter()
            _ = model(**inputs)
            if device != 'cpu' and torch.cuda.is_available():
                torch.cuda.synchronize()
            latencies.append(time.perf_counter() - start)

    latencies = np.array(latencies) * 1000  # ms
    throughput = batch_size / (np.mean(latencies) / 1000)

    return {
        'mean_latency_ms': np.mean(latencies),
        'std_latency_ms':  np.std(latencies),
        'p50_latency_ms':  np.percentile(latencies, 50),
        'p99_latency_ms':  np.percentile(latencies, 99),
        'throughput_sps':  throughput,
    }

In [ ]:
bench_device = 'cpu'
benchmark_records = []

print('Benchmarking on CPU (batch_size=1, seq_len=128)...')
baseline_bench = benchmark_inference(baseline_model, device=bench_device)
print(f'\nBaseline: {baseline_bench["mean_latency_ms"]:.1f}ms '
      f'(+/- {baseline_bench["std_latency_ms"]:.1f}ms), '
      f'{baseline_bench["throughput_sps"]:.1f} samples/s')
benchmark_records.append({'model': 'baseline', 'device': 'cpu', 'batch_size': 1, **baseline_bench})

greedy_model = apply_svd_compression(
    baseline_model, rank=ft_ranks,
    layer_names=get_target_layer_names(baseline_model),
)
greedy_bench = benchmark_inference(greedy_model, device=bench_device)
print(f'Greedy ({ft_strategy_name}): {greedy_bench["mean_latency_ms"]:.1f}ms '
      f'(+/- {greedy_bench["std_latency_ms"]:.1f}ms), '
      f'{greedy_bench["throughput_sps"]:.1f} samples/s')
benchmark_records.append({'model': f'greedy/{ft_strategy_name}',
                          'device': 'cpu', 'batch_size': 1, **greedy_bench})

speedup = baseline_bench['mean_latency_ms'] / greedy_bench['mean_latency_ms']
print(f'\nSpeedup: {speedup:.2f}x')

del greedy_model

In [ ]:
if torch.cuda.is_available():
    print('\nBenchmarking on GPU (batch_size=32, seq_len=128)...')

    baseline_gpu = benchmark_inference(baseline_model, device='cuda', batch_size=32)
    print(f'Baseline GPU: {baseline_gpu["mean_latency_ms"]:.1f}ms, '
          f'{baseline_gpu["throughput_sps"]:.0f} samples/s')
    benchmark_records.append({'model': 'baseline', 'device': 'cuda', 'batch_size': 32, **baseline_gpu})

    greedy_model = apply_svd_compression(
        baseline_model, rank=ft_ranks,
        layer_names=get_target_layer_names(baseline_model),
    )
    greedy_gpu = benchmark_inference(greedy_model, device='cuda', batch_size=32)
    print(f'Greedy GPU: {greedy_gpu["mean_latency_ms"]:.1f}ms, '
          f'{greedy_gpu["throughput_sps"]:.0f} samples/s')
    benchmark_records.append({'model': f'greedy/{ft_strategy_name}',
                              'device': 'cuda', 'batch_size': 32, **greedy_gpu})

    gpu_speedup = baseline_gpu['mean_latency_ms'] / greedy_gpu['mean_latency_ms']
    print(f'GPU Speedup: {gpu_speedup:.2f}x')

    del greedy_model
    torch.cuda.empty_cache()
else:
    print('GPU not available, skipping GPU benchmarks.')

### F.1 — Final summary figure

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel A: Pareto frontier
ax = axes[0, 0]
for stype in ['uniform', 'adaptive', 'informed', 'greedy']:
    subset = results_df[results_df['type'] == stype]
    if len(subset) == 0:
        continue
    ax.scatter(1 - subset['compression_ratio'], subset['macro_f1'],
               c=colors[stype], marker=markers[stype], s=sizes[stype],
               label=stype.replace('_', ' ').title(), alpha=0.8,
               edgecolors='white', linewidth=0.5)
ax.axhline(y=baseline_results['macro_f1'], color='green', linestyle=':', alpha=0.5)
ax.set_xlabel('Compression (1 - param ratio)')
ax.set_ylabel('Macro F1')
ax.set_title('A) Pareto Frontier')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel B: Layer importance
ax = axes[0, 1]
x = np.arange(12)
width = 0.35
ax.bar(x - width/2, attn_importance, width, label='Attention', color='steelblue', alpha=0.8)
ax.bar(x + width/2, ffn_importance,  width, label='FFN',       color='coral',     alpha=0.8)
ax.set_xlabel('Layer')
ax.set_ylabel('Importance')
ax.set_title('B) Mechanistic Layer Importance')
ax.legend(fontsize=8)
ax.set_xticks(x)

# Panel C: Recovery rates
ax = axes[1, 0]
if len(valid_recovery) > 0:
    sorted_rec = valid_recovery.sort_values('recovery_rate', ascending=True)
    colors_rec = ['red' if r < 0.5 else 'orange' if r < 0.8 else 'green'
                  for r in sorted_rec['recovery_rate']]
    ax.barh(range(len(sorted_rec)), sorted_rec['recovery_rate'].clip(-0.5, 1.5),
            color=colors_rec, alpha=0.8)
    ax.set_yticks(range(len(sorted_rec)))
    ax.set_yticklabels(sorted_rec['emotion'], fontsize=7)
    ax.axvline(x=1.0, color='green', linestyle='--', alpha=0.5)
    ax.axvline(x=0.0, color='red',   linestyle='--', alpha=0.5)
ax.set_xlabel('Recovery Rate')
ax.set_title('C) Fine-Tuning Recovery per Emotion')

# Panel D: Greedy rank allocation breakdown
ax = axes[1, 1]
mod_ranks = ft_ranks
rank_by_layer_qk  = []
rank_by_layer_v   = []
rank_by_layer_ffn = []
for l in range(12):
    qk  = [r for n, r in mod_ranks.items() if f'layer.{l}.' in n and ('query' in n or 'key' in n)]
    v   = [r for n, r in mod_ranks.items() if f'layer.{l}.' in n and ('value' in n or 'attention.output' in n)]
    ffn = [r for n, r in mod_ranks.items() if f'layer.{l}.' in n and 'attention' not in n]
    rank_by_layer_qk.append(np.mean(qk) if qk else 768)
    rank_by_layer_v.append(np.mean(v) if v else 768)
    rank_by_layer_ffn.append(np.mean(ffn) if ffn else 768)

x = np.arange(12)
w = 0.25
ax.bar(x - w, rank_by_layer_qk,  w, label='Q/K',         color='steelblue',    alpha=0.8)
ax.bar(x,     rank_by_layer_v,   w, label='V/Attn Out',  color='mediumpurple', alpha=0.8)
ax.bar(x + w, rank_by_layer_ffn, w, label='FFN',         color='coral',        alpha=0.8)
ax.set_xlabel('Layer')
ax.set_ylabel('SVD Rank')
ax.set_title(f'D) Greedy Rank Allocation ({ft_strategy_name})')
ax.legend(fontsize=8)
ax.set_xticks(x)

plt.suptitle('Informed Compression: Final Summary', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'informed_compression_summary.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## Part G — Export

All CSVs go to `results/notebook9/` for maximum information preservation.

In [ ]:
# 1. Full compression comparison table
results_df.to_csv(os.path.join(out_dir, 'compression_comparison.csv'), index=False)
print(f'Saved: compression_comparison.csv ({len(results_df)} rows)')

# 2. Per-strategy per-emotion F1 (long format)
rows_pe = []
for r in all_results:
    for emo_idx, emo in enumerate(emotion_names):
        rows_pe.append({
            'strategy': r['strategy'],
            'type': r['type'],
            'compression_ratio': r['compression_ratio'],
            'emotion': emo,
            'f1': r['per_emotion_f1'][emo_idx],
            'baseline_f1': baseline_results['per_emotion_f1'][emo_idx],
            'f1_drop': baseline_results['per_emotion_f1'][emo_idx] - r['per_emotion_f1'][emo_idx],
        })
df_pe = pd.DataFrame(rows_pe)
df_pe.to_csv(os.path.join(out_dir, 'per_emotion_per_strategy.csv'), index=False)
print(f'Saved: per_emotion_per_strategy.csv ({len(df_pe)} rows)')

# 3. Pareto frontier subset
pareto_df.to_csv(os.path.join(out_dir, 'pareto_frontier.csv'), index=False)
print(f'Saved: pareto_frontier.csv ({len(pareto_df)} rows)')

# 4. Fine-tuning recovery
recovery_df.to_csv(os.path.join(out_dir, 'finetuning_recovery.csv'), index=False)
print(f'Saved: finetuning_recovery.csv ({len(recovery_df)} rows)')

# 5. Per-emotion strategy comparison (from Part D)
if per_emotion_comparison_rows:
    df_cmp = pd.DataFrame(per_emotion_comparison_rows)
    df_cmp.to_csv(os.path.join(out_dir, 'per_emotion_strategy_at_80pct.csv'), index=False)
    print(f'Saved: per_emotion_strategy_at_80pct.csv ({len(df_cmp)} rows)')

In [ ]:
# 6. Layer importance scores (composite)
df_importance = pd.DataFrame({
    'layer': list(range(12)),
    'attn_importance': attn_importance,
    'ffn_importance': ffn_importance,
})
df_importance.to_csv(os.path.join(out_dir, 'layer_importance.csv'), index=False)
print(f'Saved: layer_importance.csv')

# 7. Informed V1 ranks per config
for config_name, ranks in informed_ranks.items():
    rank_df = pd.DataFrame([
        {'layer_name': name, 'rank': rank} for name, rank in ranks.items()
    ])
    rank_df.to_csv(os.path.join(out_dir, f'{config_name}_ranks.csv'), index=False)
print(f'Saved: {len(informed_ranks)} informed_*_ranks.csv files')

# 8. Greedy ranks per config + moves log
for config_name, ranks in greedy_ranks.items():
    rank_df = pd.DataFrame([
        {'layer_name': name, 'rank': rank} for name, rank in ranks.items()
    ])
    rank_df.to_csv(os.path.join(out_dir, f'{config_name}_ranks.csv'), index=False)
print(f'Saved: {len(greedy_ranks)} greedy_*_ranks.csv files')

# 9. Greedy moves log (full audit trail)
rows_moves = []
for config_name, info in greedy_moves_log.items():
    for m in info['moves']:
        rows_moves.append({
            'config': config_name,
            'target_ratio': info['target_ratio'],
            'achieved_ratio': info['achieved_ratio'],
            'component': m['comp'],
            'band': m['band'],
            'from_rank': m['from_rank'] if m['from_rank'] is not None else 'full',
            'to_rank': m['to_rank'],
            'saved_params': m['saved'],
            'cost_pp': m['cost'],
            'efficiency': m['eff'] if m['eff'] != float('inf') else -1,
        })
df_moves = pd.DataFrame(rows_moves)
df_moves.to_csv(os.path.join(out_dir, 'greedy_moves_log.csv'), index=False)
print(f'Saved: greedy_moves_log.csv ({len(df_moves)} rows)')

# 10. Inference benchmark results
if benchmark_records:
    df_bench = pd.DataFrame(benchmark_records)
    df_bench.to_csv(os.path.join(out_dir, 'inference_benchmarks.csv'), index=False)
    print(f'Saved: inference_benchmarks.csv ({len(df_bench)} rows)')

In [ ]:
# Final summary printout
print('\n' + '=' * 70)
print('INFORMED COMPRESSION — FINAL SUMMARY')
print('=' * 70)
print(f'Baseline: macro_f1={baseline_results["macro_f1"]:.4f}, params={baseline_params:,}')
print(f'\nBest strategy per type:')
for stype in ['uniform', 'adaptive', 'informed', 'greedy']:
    subset = results_df[results_df['type'] == stype]
    if len(subset) == 0:
        continue
    best = subset.loc[subset['macro_f1'].idxmax()]
    print(f'  {stype:15s}: {best["strategy"]:25s} -> F1={best["macro_f1"]:.4f} '
          f'({best["f1_retention"]:.1%} retained at {best["compression_ratio"]:.1%} params)')

pareto_final = compute_pareto_frontier(results_df)
n_greedy_pareto = len(pareto_final[pareto_final['type'] == 'greedy'])
print(f'\nPareto frontier: {len(pareto_final)} strategies, {n_greedy_pareto} are greedy')

print(f'\nFine-tuning recovery ({ft_strategy_name}):')
print(f'  {pre_ft_results["macro_f1"]:.4f} -> {post_ft_results["macro_f1"]:.4f} '
      f'(baseline={baseline_results["macro_f1"]:.4f})')
print(f'CPU speedup: {speedup:.2f}x')
print(f'\nAll outputs saved to: {out_dir}')